# YOLO-Pose Training Tutorial

This notebook guides you through training a YOLO-Pose model for vessel detection using either the **Large Model** or the **Small Model (Knowledge Distillation)** pipeline.

You will:
- Select which model to train (large or small)
- Review and edit the training configuration
- Launch the training script directly from the notebook
- View training logs and validation results

---

**Requirements:**
- This notebook must be run inside the Docker/Jupyter environment with all dependencies installed.
- Both training scripts and config files must be present in their respective folders.


## 1. Select Model Type

Use the dropdown below to select which model you want to train:

- **Large Model**: Standard YOLO-Pose training pipeline (no distillation)
- **Small Model (KD)**: YOLO-Pose with feature-level knowledge distillation (student-teacher setup)

**Tip:** The small model is faster and lighter, but may require a trained teacher model.


In [ ]:
import os
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# --- Model selection widget ---
model_options = {'Large Model': 'large', 'Small Model (KD)': 'small'}
model_dropdown = widgets.Dropdown(
    options=[('Large Model', 'large'), ('Small Model (KD)', 'small')],
    value='large',
    description='Model:',
    style={'description_width': 'initial'}
 )

display(model_dropdown)

# --- ABSOLUTE paths for scripts and configs (for Docker/Jupyter) ---
paths = {
    'large': {
        'script': Path('/app/large_model/train.py'),
        'config': Path('/app/large_model/config.yaml')
    },
    'small': {
        'script': Path('/app/small_model/train_KD.py'),
        'config': Path('/app/small_model/config.yaml')
    }
}

selected = model_dropdown.value
script_path = paths[selected]['script']
config_path = paths[selected]['config']

def on_model_change(change):
    global selected, script_path, config_path
    selected = change['new']
    script_path = paths[selected]['script']
    config_path = paths[selected]['config']
    clear_output(wait=True)
    display(model_dropdown)
    display(Markdown(f"**Training script:** {script_path}"))
    display(Markdown(f"**Config file:** {config_path}"))

model_dropdown.observe(on_model_change, names='value')

display(Markdown(f"**Training script:** {script_path}"))
display(Markdown(f"**Config file:** {config_path}"))

Dropdown(description='Model:', index=1, options=(('Large Model', 'large'), ('Small Model (KD)', 'small')), sty…

**Training script:** /app/small_model/train_KD.py

**Config file:** /app/small_model/config.yaml

## 2. Set Paths for Training Script and Config

Based on your selection above, the notebook will use the following files:

- **Training script**: The Python script that launches the training process
- **Config file**: The YAML file specifying all training parameters

You can review and edit the config file before starting training.


## 3. Display Selected Configuration

Below is the content of the selected config file. Review the parameters before starting training. You can edit the YAML file directly if needed.

**Key parameters:**
- `model` or `student_weights`: Path to model weights
- `data` or `yaml_file`: Path to dataset YAML
- `epochs`, `batch`, `imgsz`: Training hyperparameters
- `wandb`: Weights & Biases logging settings
- `kd`: Knowledge distillation settings (small model only)


In [2]:
import yaml
from pprint import pprint
from pathlib import Path

# Read and pretty-print the selected config file (always resolve relative to notebook location)
try:
    config_path_abs = Path(config_path).resolve()
    print(f"Resolved config path: {config_path_abs}")
    with open(config_path_abs, 'r') as f:
        config_yaml = yaml.safe_load(f)
    pprint(config_yaml)
except Exception as e:
    print(f"Error reading config: {e}")

Resolved config path: /app/small_model/config.yaml
{'datapaths': {'png_rc_patches_path': '/media/raid/opensar/vessel_rc_rescaled_patches_png',
               'rc_Ship_dataset': '/media/raid/opensar/rc_ship/',
               'rescaled_rc_patches_path': '/media/raid/opensar/range_compressed_v2/vessels/range_compressed_rescaled',
               'slc_patches_path': '/media/raid/opensar/slc_patches_png',
               'xml_labels_elong_path': '/media/ubuntu_24_04/data/opensar/Xview_Validation/vessel_label_audit/updated_xmls_shipfix_v13_RC_inflated',
               'xml_labels_path': '/media/ubuntu_24_04/data/opensar/Xview_Validation/vessel_label_audit/updated_xmls_shipfix_v10_full',
               'yolo_dataset_elong_path': '/media/raid/opensar/yolo_dataset_vessel_elong',
               'yolo_dataset_filtered_rc': '/media/raid/opensar/yolo_dataset_true_vessels_filtered',
               'yolo_dataset_full_un_filtered_rc': '/media/raid/opensar/yolo_dataset_vessel_vessel',
               'yol

## 4. Run Training Script

The cell below will launch the selected training script with the chosen config file. This may take a long time and will print logs directly in the notebook.

- For the **large model**, the script is `train.py` (run from large_model/)
- For the **small model**, the script is `../small_model/train_KD.py`

**Note:**
- Training will use the parameters from the config file you reviewed above.
- You can stop training at any time by interrupting the kernel.
- Output logs and validation results will be shown after training completes.


In [3]:
import subprocess
import sys
import os

# Build the command to run the training script
if selected == 'large':
    cmd = [sys.executable, str(script_path)]
    cwd = '/app/large_model'
else:
    cmd = [sys.executable, str(script_path), '--config', str(config_path)]
    cwd = '/app/small_model'

print(f"Running: {' '.join(str(c) for c in cmd)} (cwd={cwd})\n")
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=cwd)
for line in process.stdout:
    print(line, end='')
process.wait()
print(f"\nTraining finished with exit code {process.returncode}")

Running: /usr/bin/python3 /app/small_model/train_KD.py --config /app/small_model/config.yaml (cwd=/app/small_model)

YOLO-Pose Feature Knowledge Distillation Training
Config      : /app/small_model/config.yaml
Teacher     : /media/raid/opensar/yolo_results/runs/pose/pose/yolo26s-pose_opensar_true_vessel_filtered_dataset_640px/weights/best.pt
Student     : yolo26n-pose.pt
Dataset     : /media/raid/opensar/code/backend/pipeline/dvd_use_case/small_model/data.yaml
Run name    : kd_test_1525_01052026
Epochs      : 5
Batch size  : 16
Image size  : 640
Device      : 0
KD enabled  : True
Ultralytics 8.4.48 🚀 Python-3.12.3 torch-2.8.0a0+34c6371d24.nv25.08 CUDA:0 (NVIDIA GeForce RTX 3090, 24122MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=0.001, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/media/raid/opens

## 5. Show Training and Validation Results

After training, review the output above for logs and validation metrics. Key results to look for:

- **Box mAP@0.5**: Mean Average Precision for bounding boxes at IoU 0.5
- **Pose mAP@0.5**: Mean Average Precision for keypoints at IoU 0.5
- **Precision/Recall**: Detection and pose precision/recall

You can also check the output directory (printed in the logs) for model weights and additional results.

---

**Tip:** For more advanced analysis, you can visualize training curves and validation results using Weights & Biases (W&B) if enabled in your config.


## 6. Model Profiling: Complexity, Latency, and Resource Usage

You can profile the selected YOLO-Pose model to estimate its FLOPs, parameter count, latency, and hardware resource usage. This is useful for comparing model sizes and expected inference speed.

- The profiler uses the `profile_model` utility.
- The model weights are loaded from the `inference.model` path in the selected config.
- Results include GFLOPs, parameter count, latency, FPS, and memory usage.

**Note:** Profiling runs a batch of dummy inferences on the selected device (GPU recommended).

In [7]:
import sys
from pathlib import Path
import torch
from ultralytics import YOLO

# Add the project root to sys.path
project_root = Path('/app/large_model')  # adjust if your Docker mount is different
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utilities.read_yaml import read_yaml
from scripts.profile_model import profile_model

# Load config and get model weights path from inference section
try:
    config_path_abs = Path(config_path).resolve()
    config = read_yaml(config_path_abs)
    model_path = config['inference']['model_path']
    print(f"Profiling model weights: {model_path}")
except Exception as e:
    print(f"Error loading config or model path: {e}")
    model_path = None

if model_path:
    try:
        device = 0 if torch.cuda.is_available() else "cpu"
        model = YOLO(model_path)
        model.model = model.model.to(device)  # Ensure model is on the correct device
        stats = profile_model(model, imgsz=config['inference'].get('imgsz', 640), device=device, runs=200)
        print(stats)
    except Exception as e:
        print(f"Error profiling model: {e}")

Profiling model weights: /media/raid/opensar/yolo_results/runs/pose/yolo26n-pose_feature_kd_from_s_640_300ep_feat4/weights/best.pt



════════════════════════════════════════════════════════════
  Model Profile
════════════════════════════════════════════════════════════

  > Complexity
  ──────────────────────────────────────────────────────────
  GMACs                             3.134  GMACs
  GFLOPs                            6.268  GFLOPs
  params_M                          2.648  M params

  > Latency
  ──────────────────────────────────────────────────────────
  latency_ms                       12.152  ms
  fps                              82.288  FPS

  > GPU Memory
  ──────────────────────────────────────────────────────────
  gpu_mem_alloc_GB                  0.050  GB
  gpu_mem_peak_GB                   0.076  GB

  > GPU Compute
  ──────────────────────────────────────────────────────────
  gpu_util_avg_%                   24.750  %
  gpu_util_peak_%                  29.000  %

  > GPU Power
  ──────────────────────────────────────────────────────────
  gpu_power_avg_W                 129.737  W
  gpu_po